In [1]:
!pip install scikit-learn
!pip install tensorflow
!pip install torch

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

In [ ]:
df = pd.read_json("C:/Users/007pe/Downloads/ready_data.json", lines=True)

In [ ]:
print(df)

In [ ]:
df_copy2 = df.dropna(subset=['Speaker_party_name'])

In [ ]:
print(df_copy2)

In [ ]:
X = np.vstack(df_copy2['embedding'].values)
y = df_copy2['Speaker_party_name']

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(y) 
num_classes = len(encoder.classes_) 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0, stratify=y)

In [ ]:
hidden_units=256
dropout_rate=0.4
learning_rate=0.001

model = Sequential([
    Input(shape=(X_train.shape[1],)),  
    Dense(hidden_units, activation='relu'),  
    Dropout(dropout_rate),
    Dense(int(hidden_units / 2) , activation='relu'),  
    Dropout(dropout_rate),
    Dense(int(hidden_units / 4) , activation='relu'),  
    Dropout(dropout_rate),
    Dense(num_classes, activation='softmax')  
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
import keras_tuner as kt

def build_model(hp):
    hidden_units = hp.Int('hidden_units', min_value=32, max_value=256, step=32)
    dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4, 1e-5])
    
    model = create_model(hidden_units=hidden_units, dropout_rate=dropout_rate, learning_rate=learning_rate)
    return model

# Initialize the tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20, 
    executions_per_trial=2,  
    directory='my_tuning_dir', 
    project_name='text_classification'
)

tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Best hyperparameters:
- Hidden Units: {best_hps.get('hidden_units')}
- Dropout Rate: {best_hps.get('dropout_rate')}
- Learning Rate: {best_hps.get('learning_rate')}
""")

best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

In [ ]:
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
%%time
def train_model_with_balanced_data():
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(C=0.1, penalty='l1', solver='liblinear', max_iter=500)
    clf.fit(X_train, y_train)
    
    # Evaluate
    from sklearn.metrics import classification_report
    y_pred = clf.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    return clf

clf = train_model_with_balanced_data()

In [ ]:
svm_classifier = SVC(kernel='linear')
svm_classifier.fit(X_train, y_train)

svm_classifier_path = '/content/drive/MyDrive/Colab/svm_classifier_model.pkl'
joblib.dump(svm_classifier, svm_classifier_path)

y_pred = svm_classifier.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
rfm_classifier = RandomForestClassifier(n_estimators=100, n_jobs=3, verbose=3, random_state=0)

rfm_classifier.fit(X_train, y_train)

y_pred = rfm_classifier.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))